In [1]:
import sys
import os
import argparse

# Instead of __file__, use os.getcwd()
sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

from src.envs import N3il
from src.algos.mcts import Node

2.2.5
2.2.5


In [9]:
n=3

current_dir = os.getcwd()

args = {
    'environment': 'N3il',  # Specify the environment
    'algorithm': 'MCTS',
    'max_level_to_use_symmetry': -1,  # Use symmetry for first 2 levels (helps find compact solutions)
    'n': n,
    'C': 1.41,  # 1e-7 for n=20
    'num_searches': 100*(n**2),  # Adjusted for larger n
    'num_workers': 1,      # >1 ⇒ parallel
    'virtual_loss': 1.0,     # magnitude to subtract at reservation
    'process_bar': True,
    'display_state': True,
    'logging_mode': True,  # Enable logging mode to get return value
    'TopN': n,  # Without Priority
    "simulate_with_priority": False,
    'table_dir': current_dir,  # Directory to save tables
    'figure_dir': os.path.join(current_dir, 'figure'),  # Directory to save figures
    'random_seed': 1,  # Use the loop index as a seed for reproducibility
    'tree_visualization': False,  # Set to True to enable tree visualization
}


n3il_test = N3il((n,n), args)
state = n3il_test.get_initial_state()
node_0 = Node(n3il_test, args, state=state, parent=None, action_taken=None)
# state = n3il_test.get_next_state(state, 3)  # Example action, replace with actual action logic
# node_1 = Node(n3il_test, args, state=state, parent=node_0, action_taken=3)

In [16]:
node_0.expand().valid_moves

array([0, 1, 1, 0, 1, 0, 0, 1, 1], dtype=uint8)

In [17]:
node_0.valid_moves.reshape((n, n))

array([[0, 1, 1],
       [0, 1, 0],
       [0, 1, 1]], dtype=uint8)

In [10]:
parent_state_test = n3il_test.get_initial_state()
parent_state_test

array([[0, 0, 0],
       [0, 0, 0],
       [0, 0, 0]], dtype=uint8)

In [11]:
parent_state_test = n3il_test.get_next_state(parent_state_test, 7)  # Example action, replace with actual action logic
parent_state_test

array([[0, 0, 0],
       [0, 0, 0],
       [0, 1, 0]], dtype=uint8)

In [12]:
parent_valid_move_test = n3il_test.get_valid_moves(parent_state_test).reshape((n, n))
parent_valid_move_test

array([[1, 1, 1],
       [1, 1, 1],
       [1, 0, 1]], dtype=uint8)

In [13]:
n3il_test.get_valid_moves_subset(parent_state_test, parent_valid_move_test, 1).reshape((n, n))

array([[1, 0, 1],
       [1, 0, 1],
       [1, 0, 1]], dtype=uint8)

In [30]:
parent_state_test = n3il_test.get_next_state(parent_state_test, 3)
parent_state_test

array([[1, 1, 0],
       [1, 0, 0],
       [0, 0, 0]], dtype=uint8)

In [ ]:
n3il_test.get_valid_moves(parent_state_test).reshape((n, n))

array([[0, 0, 0],
       [0, 1, 1],
       [0, 1, 1]], dtype=uint8)

In [37]:
parent_state_test

array([[1, 1, 0],
       [0, 0, 0],
       [0, 0, 0]], dtype=uint8)

In [39]:
parent_valid_move_test

array([[0, 0, 0],
       [1, 1, 1],
       [1, 1, 1]], dtype=uint8)

In [36]:
parent_state_test.reshape(-1)

array([1, 1, 0, 0, 0, 0, 0, 0, 0], dtype=uint8)

In [ ]:
from numba import njit
@njit(cache=True, nogil=True)
def get_valid_moves_subset_nb(parent_state, parent_valid_moves, action_taken, row_count, column_count):
    """
    Given a parent state (2D boolean array) and its valid move mask (1D uint8 array),
    return a refined valid move mask for the child:
      1) Remove the action just taken.
      2) For each existing point in state, compute the line to the new point,
         then invalidate any intermediate grid points that lie exactly on that line.
      3) If slope is infinite, invalidate entire column; if slope is zero, invalidate entire row.
    Returns a flattened uint8 array: 1 = valid, 0 = invalid.
    """
    # Copy input mask and remove the taken action
    mask = parent_valid_moves.copy().reshape(-1)
    mask[action_taken] = 0

    # Coordinates of the newly placed point
    new_r = action_taken // column_count
    new_c = action_taken % column_count

    # Iterate over all existing points
    for pr in range(row_count):
        for pc in range(column_count):
            if not parent_state[pr, pc]:
                continue
            # Skip the new point itself
            if pr == new_r and pc == new_c:
                continue

            dr = pr - new_r
            dc = pc - new_c

            # Infinite slope (vertical line): invalidate entire column
            if dc == 0:
                for rr in range(row_count):
                    idx = rr * column_count + new_c
                    mask[idx] = 0
                continue

            # Zero slope (horizontal line): invalidate entire row
            if dr == 0:
                row_index = pr
                base = row_index * column_count
                for cc in range(column_count):
                    mask[base + cc] = 0
                continue

            # General (non-vertical, non-horizontal) case: remove every point on the infinite line
            # through (new_r,new_c) and (pr,pc), including both the segment and its extensions.
            for cc in range(column_count):
                # compute how far horizontally from the new point
                num = (cc - new_c) * dr
                # only those aligning to integer row are collinear
                if num % dc != 0:
                    continue
                rr = new_r + num // dc
                # skip anything outside the grid
                if rr < 0 or rr >= row_count:
                    continue
                idx = rr * column_count + cc
                mask[idx] = 0

    return mask

In [42]:
get_valid_moves_subset_nb(parent_state_test, parent_valid_move_test, 3, n, n).reshape((n, n))

array([[0, 0, 0],
       [0, 0, 0],
       [1, 1, 1]], dtype=uint8)

In [ ]:
import itertools
import numpy as np
print(np.__version__)
# np.random.seed(0)  # Removed global seed, will be set per experiment
from tqdm import trange
from numba import njit
import threading
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, wait, as_completed
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import sys
import time
import datetime
import re
from collections import defaultdict
import pprint
import math
from typing import Tuple, List, Set, Callable, NamedTuple, Union, Optional, Iterable, Dict
from multiprocessing import Pool
from sympy import Rational, Integer
from sympy.core.numbers import igcd
from src.envs import N3il, N3il_with_symmetry, supnorm_priority, supnorm_priority_array
import io
import base64
from pyvis.network import Network

import psutil
import os


def set_seeds(seed):
    """Set random seeds for reproducibility across all random number generators."""
    np.random.seed(seed)
    random.seed(seed)
    # Force compilation of numba functions with the seeded state
    # This ensures numba's internal random state is also seeded
    from numba import config
    config.THREADING_LAYER = 'safe'

@njit(cache=True, nogil=True)
def exploration_decay_nb(x):  # Monotone-down from (0,1) to (1,0)
    # Linear
    # return 1 - 0.7 * x   # Found optimal 4-point solution: 86/100 times (86.0%)
    # return 1 - x # 85/100 times (85.0%)

    # Square root (gentle early decay)
    # return 1 - 0.9 * np.sqrt(x) # 91/100 times (91.0%)
    # return 1 - 1 * np.sqrt(x) # 83/100 times (83.0%)
    # return 1 - 0.5 * np.sqrt(x) # 88/100 times (88.0%)
    # return 1 - 0.7 * np.sqrt(x) # 86%
    #return 1 - 0.8 * np.sqrt(x) # 92/100 times (92.0%)
    return 1 - 0.85 * np.sqrt(x)

    # Quadratic (faster decay)
    # return 1 - (x ** 2)

    # Exponential (custom normalization)
    # return ((np.exp(1)/(np.exp(1)-1))**2) * ((np.exp(-x)-np.exp(-1)) ** 2) # 85/100 times (85.0%)

    # Exponential fast (k=3)
    #k = 3.0
    # return (np.exp(-k * x) - np.exp(-k)) / (1 - np.exp(-k)) # 90/100 times (90.0%)

    # Exponential slow (k=1)
    # k = 1.0
    # return (np.exp(-k * x) - np.exp(-k)) / (1 - np.exp(-k)) # 86/100 times (86.0%)

    # Cosine decay
    # return 0.5 * (1 + np.cos(np.pi * x)) # 85/100 times (85.0%)

    # Rational decay
    # a = 1.0
    # return (1 - x) / (1 + a * x) # solution: 90/100 times (90.0%)

    # Logistic decay
    # k = 10.0
    # g0 = 1 / (1 + np.exp(k * (0 - 0.5)))
    # g1 = 1 / (1 + np.exp(k * (1 - 0.5)))
    # gx = 1 / (1 + np.exp(k * (x - 0.5)))
    # return (gx - g1) / (g0 - g1) # 86/100 times (86.0%)

    # Cubic decay
    # return 1 - x ** 3 # 91/100 times (91.0%)
    # return 1 - (0.9 * (x ** 3)) # 91/100 times (91.0%)

@njit(cache=True, nogil=True)
def value_fn_nb(x):
    # return x
    # return np.exp(x)
    return x

@njit(cache=True, nogil=True)
def get_value_nb(state, pts_upper_bound, value_f=value_fn_nb):
    total = np.sum(state)
    n = pts_upper_bound/2
    
    # === REVERSE REWARDING FUNCTIONS (prefer smaller point counts) ===
    
    # 1. Simple Linear Inverse: 1.0 for empty board, 0.0 for full board
    # return (n - total) / n  # Range: [0, 1]
    
    # 2. Exponential Decay (Strong preference for fewer points)
    # return np.exp(-2.0 * (total / n))  # Range: [e^-2, 1] ≈ [0.135, 1]
    # return np.exp(-1.0 * (total / n))  # Range: [e^-1, 1] ≈ [0.368, 1]
    # return np.exp(-0.5 * (total / n))  # Range: [e^-0.5, 1] ≈ [0.607, 1]
    
    # 3. Power Functions (Adjustable curvature)
    # return ((n - total) / n) ** 2  # Quadratic preference: [0, 1]
    # return ((n - total) / n) ** 0.5  # Square root preference: [0, 1]
    # return ((n - total) / n) ** 3  # Cubic preference (very aggressive): [0, 1]
    
    # 4. Sigmoid-based (Smooth transition around target)
    # target = n * 0.3  # Target 30% of grid filled
    # return 1.0 / (1.0 + np.exp(0.5 * (total - target)))  # Range: ≈[0, 1]
    # return 1.0 / (1.0 + np.exp(1.0 * (total - target)))  # Steeper transition
    
    # 5. Logarithmic Penalty
    # return max(0, 1.0 - np.log(1.0 + total) / np.log(1.0 + n))  # Range: [0, 1]
    
    # 6. ReLU-based with different thresholds
    # return max(0, (1.2 * n - total) / n)  # Reward up to 120% of n: [0, 1.2]
    # return max(0, (1.5 * n - total) / n)  # Current: reward up to 150% of n
    
    # === OPTIMAL FOR 3x3 MINIMAL COMPLETE SET (4 points) ===
    # Simple linear inverse works best for finding exact minimal sets
    return (1.6*n - total) * n  / (1.6 - 1.3)# Range: [0, 1], 1.0 for empty, 0.0 for full !!!CURRENT OPTIMAL!!!

    # Baseline rewarding function
    '''
    baseline = 1.6 * n
    theoretical_min = 1.3 * n
    num = baseline - total
    if num > 0:
        return num / (baseline - theoretical_min)  # Range: [0, 1], 1.0 for empty, 0.0 for full
    if num <= 0:
        return num / (baseline - theoretical_min)  # Range: [-1, 0], 0.0 for empty, -1.0 for full
    '''
    # Numba-safe scalar casts
    total = np.float64(np.sum(state))
    n = np.float64(pts_upper_bound) / 2.0

    # Target and normalization
    target = 0.9 * n
    max_possible = 2.0 * n
    eps = np.float64(1e-12)
    span = np.maximum(max_possible - target, eps)  # avoid division by zero
    # Normalized distance: 0 at target, 1 at 2n (can be < 0 if total < target)
    tnorm = (total - target) / span

    # ---- Choose ONE of the following returns (uncomment exactly one) ----

    # 2) Quadratic (penalizes farther from target more strongly)
    # return np.clip(1.0 - tnorm * tnorm, 0.0, 1.0)

    # 3) Gaussian peak at target (default active; sharp pull to 0.9n)
    # sigma = np.maximum(0.05 * n, eps)  # controls sharpness
    # return np.exp(-0.5 * ((total - target) / sigma) ** 2)

    # 4) Logistic decay from target upward
    # k = 6.0 / np.maximum(n, 1.0)
    # return 1.0 / (1.0 + np.exp(k * (total - target)))

    # 5) Rational distance penalty (gentler tail)
    # alpha = 2.0 / np.maximum(n, 1.0)
    # return 1.0 / (1.0 + alpha * np.abs(total - target))

    # 6) Piecewise: full at/below target, then linear drop to 0 at 2n
    # if total <= target:
    #     return 1.0
    # else:
    #     return np.maximum(0.0, 1.0 - (total - target) / span)

    # 7) Cosine half-wave on [target, 2n] (smooth with zero slope at target)
    # x = np.clip(tnorm, 0.0, 1.0)               # map [target,2n] -> [0,1]
    # return 0.5 * (1.0 + np.cos(np.pi * x))     # 1 at target, 0 at 2n

    # ------------ Positive-direction variants (optimum at 2n) ------------
    # Use these if you want to test the opposite objective (larger total better).
    # 1+) Linear increasing from target to 2n
    # return np.clip(tnorm, 0.0, 1.0)

    # 2+) Quadratic increasing (slow start, faster near 2n)
    # x = np.clip(tnorm, 0.0, 1.0)
    # return x * x

    # 3+) Exponential rise (very low until near 2n)
    # x = np.clip(tnorm, 0.0, 1.0)
    # k = 4.0
    # return (np.exp(k * x) - 1.0) / (np.exp(k) - 1.0)

# JIT-compiled function to check if three points are collinear
@njit(cache=True, nogil=True)
def _are_collinear(x1, y1, x2, y2, x3, y3):
    return (y1 - y2) * (x1 - x3) == (y1 - y3) * (x1 - x2)

# JIT-compiled function to determine valid moves on the board 
@njit(cache=True, nogil=True)
def get_valid_moves_nb(state, row_count, column_count):
    max_pts = row_count * column_count
    coords = np.empty((max_pts, 2), np.int64)
    n_pts = 0

    # Collect coordinates of existing points
    for i in range(row_count):
        for j in range(column_count):
            if state[i, j] == 1:
                coords[n_pts, 0] = i
                coords[n_pts, 1] = j
                n_pts += 1

    mask = np.zeros(row_count * column_count, np.uint8)

    # Check each empty cell
    for i in range(row_count):
        for j in range(column_count):
            if state[i, j] != 0:
                continue
            valid = True
            # Check for collinearity with every pair of existing points
            for p in range(n_pts):
                for q in range(p + 1, n_pts):
                    i1, j1 = coords[p, 0], coords[p, 1]
                    i2, j2 = coords[q, 0], coords[q, 1]
                    if _are_collinear(j1, i1, j2, i2, j, i):
                        valid = False
                        break
                if not valid:
                    break
            if valid:
                mask[i * column_count + j] = 1
    return mask

@njit(cache=True, nogil=True)
def get_valid_moves_subset_nb(parent_state, parent_valid_moves, action_taken, row_count, column_count):
    """
    Given a parent state (2D boolean array) and its valid move mask (1D uint8 array),
    return a refined valid move mask for the child:
      1) Remove the action just taken.
      2) For each existing point in state, compute the line to the new point,
         then invalidate any intermediate grid points that lie exactly on that line.
      3) If slope is infinite, invalidate entire column; if slope is zero, invalidate entire row.
    Returns a flattened uint8 array: 1 = valid, 0 = invalid.
    """
    # Copy input mask and remove the taken action
    mask = parent_valid_moves.copy()
    mask[action_taken] = 0

    # Coordinates of the newly placed point
    new_r = action_taken // column_count
    new_c = action_taken % column_count

    # Iterate over all existing points
    for pr in range(row_count):
        for pc in range(column_count):
            if not parent_state[pr, pc]:
                continue
            # Skip the new point itself
            if pr == new_r and pc == new_c:
                continue

            dr = pr - new_r
            dc = pc - new_c

            # Infinite slope (vertical line): invalidate entire column
            if dc == 0:
                for rr in range(row_count):
                    idx = rr * column_count + new_c
                    mask[idx] = 0
                continue

            # Zero slope (horizontal line): invalidate entire row
            if dr == 0:
                row_index = pr
                base = row_index * column_count
                for cc in range(column_count):
                    mask[base + cc] = 0
                continue

            # General (non-vertical, non-horizontal) case: remove every point on the infinite line
            # through (new_r,new_c) and (pr,pc), including both the segment and its extensions.
            for cc in range(column_count):
                # compute how far horizontally from the new point
                num = (cc - new_c) * dr
                # only those aligning to integer row are collinear
                if num % dc != 0:
                    continue
                rr = new_r + num // dc
                # skip anything outside the grid
                if rr < 0 or rr >= row_count:
                    continue
                idx = rr * column_count + cc
                mask[idx] = 0

    return mask

# JIT-compiled function to count collinear triples on the board
@njit(cache=True, nogil=True)
def check_collinear_nb(state, row_count, column_count):
    max_pts = row_count * column_count
    coords = np.empty((max_pts, 2), np.int64)
    n_pts = 0

    # Collect all placed point coordinates
    for i in range(row_count):
        for j in range(column_count):
            if state[i, j] == 1:
                coords[n_pts, 0] = i
                coords[n_pts, 1] = j
                n_pts += 1

    triples = 0
    # Count all collinear triplets
    for a in range(n_pts):
        for b in range(a + 1, n_pts):
            for c in range(b + 1, n_pts):
                i1, j1 = coords[a, 0], coords[a, 1]
                i2, j2 = coords[b, 0], coords[b, 1]
                i3, j3 = coords[c, 0], coords[c, 1]
                if _are_collinear(j1, i1, j2, i2, j3, i3):
                    triples += 1
    return triples

@njit(cache=True, nogil=True)
def simulate_nb(state, row_count, column_count, pts_upper_bound):
    """
    Perform random rollout until no valid moves remain.
    Return normalized value using a custom value function.
    Uses get_valid_moves_subset_nb for incremental validity updates.
    Note: This function uses numba's random number generator which is seeded globally.
    """
    max_size = row_count * column_count
    # Initial valid moves mask
    valid_moves = get_valid_moves_nb(state, row_count, column_count)
    total_valid = np.sum(valid_moves)

    while total_valid > 0:
        # Build list of valid actions
        acts = np.empty(total_valid, np.int64)
        k = 0
        for idx in range(max_size):
            if valid_moves[idx]:
                acts[k] = idx
                k += 1
        # Randomly select one valid action and place the point
        pick = acts[np.random.randint(0, total_valid)]

        # Incrementally update valid_moves using subset-based filtering
        valid_moves = get_valid_moves_subset_nb(
            state,
            valid_moves,
            pick,
            row_count,
            column_count
        )

        r = pick // column_count
        c = pick % column_count
        state[r, c] = 1  # mark the new point

        total_valid = np.sum(valid_moves)

    # Compute and return the final value
    return get_value_nb(state, pts_upper_bound)

@njit(cache=True, nogil=True)
def filter_top_priority_moves(valid_moves, priority_grid, row_count, column_count, top_N=1):
    """
    Numba-accelerated: Filter valid moves to only those with the top_N highest priorities.

    Args:
        valid_moves (np.ndarray): 1D array (flattened) of valid moves (1=valid, 0=invalid).
        priority_grid (np.ndarray): 2D array of priority values for each grid cell.
        row_count (int): Number of rows in the grid.
        column_count (int): Number of columns in the grid.
        top_N (int): Number of top priority levels to select.

    Returns:
        np.ndarray: 1D mask array with only the top_N-priority valid moves set to 1.
    """
    indices = []
    priorities = []
    for idx in range(valid_moves.shape[0]):
        if valid_moves[idx] == 1:
            indices.append(idx)
            i = idx // column_count
            j = idx % column_count
            priorities.append(priority_grid[i, j])
    if len(indices) == 0:
        return valid_moves

    # Find the unique priorities and sort descending
    # Numba doesn't support np.unique or sort for lists, so do it manually
    # 1. Copy priorities to a new array
    n = len(priorities)
    unique_priorities = []
    for k in range(n):
        p = priorities[k]
        found = False
        for l in range(len(unique_priorities)):
            if unique_priorities[l] == p:
                found = True
                break
        if not found:
            unique_priorities.append(p)
    # 2. Sort unique_priorities descending (simple selection sort)
    for i in range(len(unique_priorities)):
        max_idx = i
        for j in range(i+1, len(unique_priorities)):
            if unique_priorities[j] > unique_priorities[max_idx]:
                max_idx = j
        # Swap
        tmp = unique_priorities[i]
        unique_priorities[i] = unique_priorities[max_idx]
        unique_priorities[max_idx] = tmp

    # 3. Select top_N priorities
    N = min(top_N, len(unique_priorities))
    threshold = unique_priorities[:N]

    # 4. Build mask
    mask = np.zeros_like(valid_moves)
    for k in range(n):
        idx = indices[k]
        p = priorities[k]
        for t in range(N):
            if p == threshold[t]:
                mask[idx] = 1
                break
    return mask

@njit(cache=True, nogil=True)
def simulate_with_priority_nb(state, row_count, column_count, pts_upper_bound, priority_grid, top_N):
    """
    Perform a random rollout that first filters valid moves by priority
    and then proceeds like simulate_nb, but initial valid moves are pre-filtered.
    Args:
        state (np.ndarray): 2D board state.
        row_count (int): Number of rows.
        column_count (int): Number of columns.
        pts_upper_bound (int): Scoring upper bound.
        priority_grid (np.ndarray): 2D array of priorities.
        top_N (int): Number of top priority levels to keep.
    Returns:
        float: Normalized final value.
    """
    max_size = row_count * column_count

    # Initial valid moves mask
    valid_moves = get_valid_moves_nb(state, row_count, column_count)
    # Pre-filter by priority
    valid_moves = filter_top_priority_moves(
        valid_moves, priority_grid, row_count, column_count, top_N
    )
    total_valid = np.sum(valid_moves)

    # Rollout until no moves remain
    while total_valid > 0:
        acts = np.empty(total_valid, np.int64)
        k = 0
        for idx in range(max_size):
            if valid_moves[idx]:
                acts[k] = idx
                k += 1

        pick = acts[np.random.randint(0, total_valid)]

        # Update valid moves and state
        valid_moves = get_valid_moves_subset_nb(
            state, valid_moves, pick, row_count, column_count
        )
        state[pick // column_count, pick % column_count] = 1

        # Filter again by priority
        valid_moves = filter_top_priority_moves(
            valid_moves, priority_grid, row_count, column_count, top_N
        )
        total_valid = np.sum(valid_moves)

    return get_value_nb(state, pts_upper_bound)


class Node:
    def __init__(self, game, args, state, parent=None, action_taken=None):
        self.game = game
        self.args = args
        self.state = state
        self.parent = parent
        self.action_taken = action_taken

        self.children = []
        self.visit_count = 0
        self.value_sum = 0
        self.lock = threading.Lock()
        self._vl = args.get('virtual_loss', 1.0)

        if parent is None:
            self.level = np.sum(state)  # Level is the number of points placed
            if (self.level <= game.max_level_to_use_symmetry and 
                hasattr(game, 'get_valid_moves_with_symmetry')):
                self.valid_moves = game.get_valid_moves_with_symmetry(state)
            else:
                self.valid_moves = game.get_valid_moves(state)
        else:
            self.level = parent.level + 1
            if (self.level <= game.max_level_to_use_symmetry and 
                hasattr(game, 'get_valid_moves_subset_with_symmetry')):
                self.valid_moves = game.get_valid_moves_subset_with_symmetry(
                    parent.state, parent.action_space, self.action_taken)
            else:
                self.valid_moves = game.get_valid_moves_subset(
                    parent.state, parent.action_space, self.action_taken)

        self.action_space = self.valid_moves.copy()
        self.action_space.flags.writeable = False

        self.is_full = False
        self._cached_ucb = None     # Cached UCB value
        self._ucb_dirty = True      # Indicates whether the cached UCB is stale

    def apply_virtual_loss(self):
        with self.lock:
            self.value_sum -= self._vl
            self.visit_count += 1
            self._ucb_dirty = True  # Mark UCB as outdated

    def revert_virtual_loss(self):
        with self.lock:
            self.value_sum += self._vl
            self._ucb_dirty = True  # Mark UCB as outdated

    def is_fully_expanded(self):
        return self.is_full and len(self.children) > 0

    def select(self, iter):
        best_child = None
        best_ucb = -np.inf
        log_N = math.log(self.visit_count)

        for child in self.children:
            ucb = self.get_ucb(child, iter, log_N)
            if ucb > best_ucb:
                best_child = child
                best_ucb = ucb

        return best_child

    def get_ucb(self, child, iter, log_N=None):
        if log_N is None:
            log_N = math.log(self.visit_count)

        with child.lock:
            if not child._ucb_dirty and child._cached_ucb is not None:
                return child._cached_ucb

            q_value = child.value_sum / child.visit_count
            T_i = self.args['C'] * exploration_decay_nb(iter/self.args['num_searches'])
            exploration_value = T_i * math.sqrt(log_N / child.visit_count)
            ucb = q_value + exploration_value
            # print("Exploit:", q_value)
            # print("Explore:", exploration_value)
            child._cached_ucb = ucb
            child._ucb_dirty = False
            return ucb

    def expand(self):
        valid_indices = np.where(self.valid_moves == 1)[0]
        action = np.random.choice(valid_indices)
        self.valid_moves[action] = 0

        if np.sum(self.valid_moves) == 0:
            self.is_full = True

        child_state = self.state.copy()
        child_state = self.game.get_next_state(child_state, action)

        child = Node(self.game, self.args, child_state, self, action)
        self.children.append(child)
        return child

    def simulate(self):
        tmp = self.state.copy()
        if self.args["simulate_with_priority"] == True:
            return simulate_with_priority_nb(tmp,
                                            self.game.row_count,
                                            self.game.column_count,
                                            self.game.pts_upper_bound,
                                            self.game.priority_grid,
                                            self.args['TopN'])
        else:
            return simulate_nb(tmp,
                            self.game.row_count,
                            self.game.column_count,
                            self.game.pts_upper_bound)

    def backpropagate(self, value):
        with self.lock:
            self.value_sum += value
            self._ucb_dirty = True  # Mark UCB as outdated
        self.visit_count += 1
        if self.parent is not None:
            self.parent.backpropagate(value)

2.2.5
